In [ ]:
import pandas as pd
import numpy as np
import os 
import matplotlib.pyplot as plt
import pypsa
import yaml

### Scenario selection

In [ ]:
PREFIX = "/mnt/c/Users/scl38887/Documents/git/pypsa-nz/"

In [ ]:
OUTPUT = PREFIX + "results/figures/sensitivity/"

In [ ]:
scenarios = {#"0": PREFIX + f"results/nz_2030_exp/postnetworks/elec_s_10_ec_lc3.0_CO2L_3H_2030_0.071_AB_0export.nc",
            "0": PREFIX + f"results/nz_2035_lowfix_lowcagr/sensitivity/postnetworks/elec_s_10_ec_lc3.0_CO2L_3H_2035_0.071_BU_10export_sens0.nc",
            "30": PREFIX + f"results/nz_2035_lowfix_lowcagr/sensitivity/postnetworks/elec_s_10_ec_lc3.0_CO2L_3H_2035_0.071_BU_10export_sens30.nc",
            "50": PREFIX + f"results/nz_2035_lowfix_lowcagr/sensitivity/postnetworks/elec_s_10_ec_lc3.0_CO2L_3H_2035_0.071_BU_10export_sens50.nc",
            }

In [ ]:
tech_colors = PREFIX + "config/tech_colors.yaml"

In [ ]:
threshold_dispatch = 1e6 # TWh

### Validation

In [ ]:
# Compare the network solved using the sensitivity workflow vs the regular workflow

n_regular = pypsa.Network(PREFIX + "results/nz_2035_lowfix_lowcagr/postnetworks/elec_s_10_ec_lc3.0_CO2L_3H_2035_0.071_BU_10export.nc")

n_sens0 = pypsa.Network(PREFIX + "results/nz_2035_lowfix_lowcagr/sensitivity/postnetworks/elec_s_10_ec_lc3.0_CO2L_3H_2035_0.071_BU_10export_sens0.nc")

n_sens30 = pypsa.Network(PREFIX + "results/nz_2035_lowfix_lowcagr/sensitivity/postnetworks/elec_s_10_ec_lc3.0_CO2L_3H_2035_0.071_BU_10export_sens30.nc")

#### Cost difference electrolyzer

In [ ]:
ely_cost_regular = n_regular.links[n_regular.links.carrier == "H2 Electrolysis"].capital_cost.mean()

In [ ]:
ely_cost_sens30 = n_sens30.links[n_sens30.links.carrier == "H2 Electrolysis"].capital_cost.mean()

In [ ]:
print(f"The cost difference between the regular and sensitivity (30) networks is ", ((ely_cost_sens30 - ely_cost_regular) / ely_cost_regular) * 100, "%")

#### Objective value

In [ ]:
n_regular.objective / 1e6, n_sens0.objective / 1e6, n_sens30.objective / 1e6

#### Hydrogen price

In [ ]:
get_weighted_hydrogen_price(n_regular)

In [ ]:
get_weighted_hydrogen_price(n_sens0)

#### Dispatch

In [ ]:
(n_regular.statistics().loc["Generator",:]/1e6).round(1)

In [ ]:
(n_sens0.statistics().loc["Generator",:]/1e6).round(1)

### Data preparation

In [ ]:
def get_weighted_hydrogen_price(n):

    # Get H2 buses
    buses_h2 = n.buses[n.buses.carrier == "H2"].index

    hydrogen_price = n.buses_t.marginal_price[buses_h2]

    # hydrogen balance for H2 buses: set any positive values (net exports) to zero so only demand remains
    h2_balance = n.statistics.energy_balance(aggregate_time=False, aggregate_bus=False).loc[:,:,"H2",buses_h2]
    hydrogen_demand = h2_balance.clip(upper=0)
    hydrogen_demand = hydrogen_demand.groupby("bus").sum().T

    # Weighted average over location and time
    weighted_hydrogen_price = (hydrogen_price * hydrogen_demand).mean() / hydrogen_demand.mean()
    weighted_hydrogen_price = weighted_hydrogen_price.mean()

    return weighted_hydrogen_price

In [ ]:
def get_weighted_electricity_price(n):

    # Get AC buses
    buses_ac = n.buses[n.buses.carrier == "AC"].index

    electricity_price = n.buses_t.marginal_price[buses_ac]

    # electricity balance for AC buses: set any positive values (net exports) to zero so only demand remains
    el_balance = n.statistics.energy_balance(aggregate_time=False, aggregate_bus=False).loc[:,:,"AC",buses_ac]
    electricity_demand = el_balance.clip(upper=0)
    electricity_demand = electricity_demand.groupby("bus").sum().T

    # Weighted average over location and time
    weighted_electricity_price = (electricity_price * electricity_demand).mean() / electricity_demand.mean()
    weighted_electricity_price = weighted_electricity_price.mean()

    return weighted_electricity_price

In [ ]:
def get_res_share(dispatch, threshold_dispatch_sum):

    res_techs = [
    "Geothermal",
    "Offshore Wind (AC)",
    "Offshore Wind (DC)",
    "Onshore Wind",
    "Run of River",
    "Solar",
    "Reservoir & Dam",
    ]

    fossil_techs = [
        "Coal",
        "Combined-Cycle Gas",
        "Open-Cycle Gas",
        "urban central gas CHP",
        "Oil",
        "urban central solid biomass CHP CC" # really?
    ]

    dispatch = dispatch.T

    re = dispatch[dispatch.index.isin(res_techs)].sum().values[0]
    fossil = dispatch[dispatch.index.isin(fossil_techs)].sum().values[0]

    # Check, if re + fossil = total dispatch - threshold
    if abs(dispatch.sum().values[0] - threshold_dispatch_sum - re - fossil) > 1000:
        raise ValueError("Sum of dispatches does not match threshold dispatch sum. RE: " + str(re/1e6) + " Fossil: " + str(fossil/1e6) + " Total: " + str(dispatch.sum().values[0]/1e6 - threshold_dispatch_sum/1e6))

    res_share = re / (re + fossil)


    return res_share

In [ ]:
def get_emissions(dispatch, n):

    efficiency = n.generators.groupby("carrier")["efficiency"].mean()
    efficiency.index = n.carriers.loc[efficiency.index].nice_name.values 

    e_factor = n.carriers
    e_factor.index = e_factor["nice_name"]
    e_factor.index.name = "carrier"
    e_factor = e_factor.loc[:, "co2_emissions"]
    e_factor = e_factor[~e_factor.index.duplicated(keep='first')] # Delete values without index of e_factor

    emissions = dispatch.T.iloc[:,0] * e_factor  * (1/efficiency)

    emissions = emissions.sum() / 1e6 # in MtCO2
    
    return emissions.sum()

In [ ]:
def get_analysis(show_demand=True):
    """Get dispatch
    """

    dispatch_all = pd.DataFrame()
    res_share_all = pd.DataFrame()
    emissions_all = pd.DataFrame()
    objective_all = pd.DataFrame()
    electricity_price_all = pd.DataFrame()
    hydrogen_price_all = pd.DataFrame()

    for sc in scenarios:

        n = pypsa.Network(scenarios[sc])

        dispatch = n.statistics.dispatch(bus_carrier="AC")[n.statistics.dispatch(bus_carrier="AC") > 0]
        threshold_dispatch_sum = dispatch[dispatch < threshold_dispatch].sum()
        dispatch = dispatch[dispatch > threshold_dispatch]
        dispatch = dispatch.groupby(level=1).sum() # Combine links and generators, e.g. for OCGT
        dispatch = pd.DataFrame(dispatch).T
        dispatch[f"(Dispatch < {threshold_dispatch/1e6} TWh thres.)"] = threshold_dispatch_sum

        dispatch.index = [sc]

        dispatch_all = pd.concat([dispatch_all, dispatch], axis=0) #, ignore_index=True)
        res_share_all = pd.concat([res_share_all, pd.DataFrame([get_res_share(dispatch, threshold_dispatch_sum)], index=[sc])], axis=0)
        emissions_all = pd.concat([emissions_all, pd.DataFrame([get_emissions(dispatch, n)], index=[sc])], axis=0)

        objective_all = pd.concat([objective_all, pd.DataFrame([n.objective], index=[sc])], axis=0)
        electricity_price_all = pd.concat([electricity_price_all, pd.DataFrame([get_weighted_electricity_price(n)], index=[sc])], axis=0)
        hydrogen_price_all = pd.concat([hydrogen_price_all, pd.DataFrame([get_weighted_hydrogen_price(n)], index=[sc])], axis=0)


    return objective_all, electricity_price_all, hydrogen_price_all, res_share_all, dispatch_all, emissions_all

#### Analysis

In [ ]:
objective, electricity_price, hydrogen_price, res_share, dispatch, emissions = get_analysis(show_demand=False)

In [ ]:
objective, electricity_price, hydrogen_price

In [ ]:
res_share

In [ ]:
emissions

In [ ]:
dispatch

#### Plot

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Example input dataframes (each has index = electrolyser cost change [%])
emissions_test = emissions.rename(columns={0: 'value'})
h2_price_test   = hydrogen_price.rename(columns={0: 'value'})
re_share_test   = res_share.rename(columns={0: 'value'})

# Combine into dictionary for convenience
dfs = {
    "Emissions": emissions_test,
    "Hydrogen price": h2_price_test,
    "RE share": re_share_test   
}

# Compute relative (% change) compared to baseline (index 0)
baseline_index = "0"  # electrolyser cost = 0%
for name, df in dfs.items():
    base_val = df.loc[baseline_index, 'value']
    df['change_%'] = (df['value'] / base_val - 1) * 100

# Plot
plt.figure(figsize=(7, 4))
for name, df in dfs.items():
    plt.plot(df.index, df['change_%'], marker='o', label=name)

plt.axhline(0, color='gray', linewidth=1, linestyle='--')
plt.xlabel("Electrolyser capital cost change in %")
plt.ylabel("Change in indicator in %")
plt.title("Sensitivity of key indicators to electrolyser cost changes")
plt.legend()
plt.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()
plt.show()


### Misc

In [ ]:
n_sens0.statistics.capex() #.sum()

In [ ]:
n_sens30.statistics.capex() #.sum()

In [ ]:
threshold = 1e5
diff = n_sens30.statistics.capex() - n_sens0.statistics.capex()
diff[(diff < -threshold) | (diff > threshold)] / 1e6

#### Installed capacity in GW

In [ ]:
threshold = 1e2
diff = n_sens30.statistics.optimal_capacity() - n_sens0.statistics.optimal_capacity()
(diff[(diff < -threshold) | (diff > threshold)] / 1e3).round(1)

In [ ]:
n_sens0.statistics.optimal_capacity().loc[:, "H2 Electrolysis"] / 1e3

In [ ]:
n_sens30.statistics.optimal_capacity().loc[:, "H2 Electrolysis"] / 1e3

#### Dispatch in TWh

In [ ]:
threshold = 1e5
diff = n_sens30.statistics.dispatch() - n_sens0.statistics.dispatch()
diff[(diff < -threshold) | (diff > threshold)] / 1e6

#### RE share